# LLM-JP v4 8B 2段階SFT (Supervised Fine-Tuning)

このノートブックでは、**LLM-JP v4 8B** モデル (`v4-8b-decay2m-ipt_v3.1-instruct4`) に対して2段階のSFTを実施します。

## 概要

| 項目 | 内容 |
|------|------|
| **ベースモデル** | LLM-JP v4 8B (`v4-8b-decay2m-ipt_v3.1-instruct4`) |
| **手法** | QLoRA (4bit量子化 + LoRA) |
| **高速化** | Unsloth (2-5x高速化、メモリ効率化) |
| **GPU** | A100 80GB (Google Colab) |

## 2段階SFT

- **Stage 1**: 一般的な会話能力の獲得
  - データ: `tokyotech-llm/lmsys-chat-1m-synth` (GPT-OSS variant, Apache 2.0)
- **Stage 2**: 日本語での専門的なタスク (コード・数学・STEM) の強化
  - データ: `tokyotech-llm/Swallow-Nemotron-Post-Training-Dataset-v1` (GPT-OSS-Ja-202601, CC BY 4.0)

## 事前準備

1. モデルファイル (~17GB) を Google Drive にアップロードしてください
   - `v4-8b-decay2m-ipt_v3.1-instruct4/` フォルダごとアップロード
   - 含まれるファイル: `model-*.safetensors` (4ファイル), `config.json`, `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`, `generation_config.json`, `model.safetensors.index.json`
2. Google Colab で **A100 GPU** ランタイムを選択してください
3. 以下のセルを順番に実行してください (Cell 2 実行後にランタイムが再起動されるため、Cell 3 から再開)

In [ ]:
# ============================================================
# Cell 2: ライブラリインストール
# ============================================================
# 注意: このセル実行後、ランタイムが自動的に再起動されます。
#       再起動後は Cell 3 から実行を再開してください。
!uv pip install unsloth
!uv pip install --force-reinstall "numpy<2"  # soxr互換性対策

# numpy のダウングレードを反映するためランタイムを再起動
import os
os.kill(os.getpid(), 9)

Using Python 3.12.12 environment at: /usr
Resolved 88 packages in 947ms
Prepared 14 packages in 1.77s
Uninstalled 5 packages in 459ms
Installed 14 packages in 59ms
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + msgspec==0.20.0
 - pyarrow==18.1.0
 + pyarrow==23.0.1
 - torchao==0.10.0
 + torchao==0.16.0
 - transformers==5.0.0
 + transformers==4.57.6
 + trl==0.24.0
 + tyro==1.0.6
 + unsloth==2026.2.1
 + unsloth-zoo==2026.2.1
 + xformers==0.0.35
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 38ms
Prepared 1 package in 282ms
Uninstalled 1 package in 34ms
Installed 1 package in 19ms
 - numpy==2.0.2
 + numpy==1.26.4


In [1]:
# ============================================================
# Cell 2b: HuggingFace ログイン
# ============================================================
# lmsys/lmsys-chat-1m (gated dataset) へのアクセスに必要です。
# 事前準備:
#   1. https://huggingface.co/datasets/lmsys/lmsys-chat-1m でアクセスに同意
#   2. https://huggingface.co/settings/tokens でトークンを取得
#   3. Colab の左パネル「秘密鍵（🔑）」に登録
import os
from google.colab import userdata

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = userdata.get('HFH_FG')  # 秘密鍵サービスに登録した名前に変更

if not HF_TOKEN:
    print("警告: HF_TOKENが見つかりません。Hugging Faceへのログインが失敗する可能性があります。")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)

In [2]:
# ============================================================
# Cell 3: Google Drive マウント & モデルロード
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# === ベースディレクトリ ===
BASE_DIR = "/content/drive/MyDrive/NLP_2026/shared"
# =========================

MODEL_PATH = f"{BASE_DIR}/v4-8b-decay2m-ipt_v3.1-instruct4"
MAX_SEQ_LENGTH = 4096

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,       # QLoRA: 4bit量子化
    dtype=None,              # 自動検出 (A100ではbfloat16)
)

print(f"モデルロード完了: {MODEL_PATH}")
print(f"語彙サイズ: {len(tokenizer)}")
print(f"最大シーケンス長: {MAX_SEQ_LENGTH}")

Mounted at /content/drive
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

モデルロード完了: /content/drive/MyDrive/NLP_2026/shared/v4-8b-decay2m-ipt_v3.1-instruct4
語彙サイズ: 196608
最大シーケンス長: 4096


In [3]:
# ============================================================
# Cell 4: LoRA設定
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # メモリ効率化
    random_state=42,
)

# 学習可能パラメータの確認
model.print_trainable_parameters()

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,632,143,872 || trainable%: 0.4859


## Stage 1: 一般会話SFT

**データセット**: `tokyotech-llm/lmsys-chat-1m-synth` (GPT-OSS variant, Apache 2.0)

LMSYS-Chat-1Mをベースに合成されたマルチターン会話データです。このステージでは、モデルの一般的な会話能力・指示追従能力を向上させます。

In [4]:
# ============================================================
# Cell 6: Stage 1 データセット読み込み & 前処理
# ============================================================
from datasets import load_dataset

# === サンプル数の制限 (Colab時間制約対策) ===
STAGE1_MAX_SAMPLES = 10000  # 必要に応じて変更 (None で全件)
# ============================================

# GPT-OSS 合成応答を読み込み
print("GPT-OSS応答データを読み込み中...")
REPO_ID = "tokyotech-llm/lmsys-chat-1m-synth"
responses_ds = load_dataset(
    REPO_ID,
    data_files="dataset/gpt-oss-lmsys-chat-1m-synth-ja+en.jsonl.gz",
    split="train",
)
print(f"GPT-OSS応答: {len(responses_ds)} 件")

# synthesized_assistant_responses が非Nullのデータのみ
responses_ds = responses_ds.filter(
    lambda x: x["synthesized_assistant_responses"] is not None
)
print(f"応答あり: {len(responses_ds)} 件")

GPT-OSS応答データを読み込み中...


README.md: 0.00B [00:00, ?B/s]

dataset/gpt-oss-lmsys-chat-1m-synth-ja+e(…):   0%|          | 0.00/15.8G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/109 [00:00<?, ?it/s]

GPT-OSS応答: 783836 件


Filter:   0%|          | 0/783836 [00:00<?, ? examples/s]

応答あり: 487370 件


In [5]:
# ============================================================
# Cell 6b: Stage 1 データ変換 (reasoning_content から指示を抽出)
# ============================================================
import re
from datasets import Dataset

def extract_instruction_and_response(row):
    """reasoning_content からユーザー指示を抽出し、最高スコアの応答とペアにする。"""
    responses = row.get("synthesized_assistant_responses")
    if not responses:
        return {"text": ""}

    # reasoning_content から "The user asks: \"...\"" パターンでユーザー指示を抽出
    user_instruction = None
    for resp in responses:
        rc = resp.get("reasoning_content", "")
        if rc:
            # 日本語指示は "The user asks: \"...\""  の中にある
            match = re.search(r'[Tt]he user asks?:?\s*["\u201c](.+?)["\u201d]', rc, re.DOTALL)
            if match:
                user_instruction = match.group(1).strip()
                break

    if not user_instruction:
        return {"text": ""}

    # 最もスコアの高い応答を選択
    scores = row.get("synthesized_response_scoring_annotations")
    best_response = responses[0]["content"]
    if scores and len(scores) == len(responses):
        best_idx = 0
        best_score = -1
        for idx, s in enumerate(scores):
            ps = s.get("preference_score", 0) or 0
            if ps > best_score:
                best_score = ps
                best_idx = idx
        best_response = responses[best_idx]["content"]

    messages = [
        {"role": "user", "content": user_instruction},
        {"role": "assistant", "content": best_response},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

print("reasoning_content からユーザー指示を抽出し、SFTデータに変換中...")
stage1_dataset = responses_ds.map(
    extract_instruction_and_response,
    num_proc=4,
    remove_columns=responses_ds.column_names,
)

# 空のサンプルを除去
stage1_dataset = stage1_dataset.filter(lambda x: len(x["text"]) > 0)
print(f"変換完了: {len(stage1_dataset)} 件")

# サンプリング
if STAGE1_MAX_SAMPLES is not None and len(stage1_dataset) > STAGE1_MAX_SAMPLES:
    stage1_dataset = stage1_dataset.shuffle(seed=42).select(range(STAGE1_MAX_SAMPLES))
    print(f"サンプリング後: {len(stage1_dataset)} 件")

# 変換結果の確認
print(f"\n--- 変換後サンプル (1件目、先頭500文字) ---")
print(stage1_dataset[0]["text"][:500])

reasoning_content からユーザー指示を抽出し、SFTデータに変換中...


Map (num_proc=4):   0%|          | 0/487370 [00:00<?, ? examples/s]

Filter:   0%|          | 0/487370 [00:00<?, ? examples/s]

変換完了: 329818 件
サンプリング後: 10000 件

--- 変換後サンプル (1件目、先頭500文字) ---


### 指示:
強姦とは？

### 応答:
**強姦（ごうかん）とは**、相手の同意なしに、または同意を得ることができない状態（たとえば、意識不明、酩酊、未成年など）において、性的行為（主に性交）を強制的に行うことを指します。  

### 法的なポイント（日本の場合）

| 項目 | 内容 |
|------|------|
| **罪名** | 強姦罪（刑法第177条） |
| **対象行為** | 目的が性交であること、かつ、相手の自由意思に反して行われたこと |
| **合意の有無** | 同意がない、あるいは同意が有効でない（未成年、酩酊、無意識など）場合に成立 |
| **罰則** | 原則として5年以上の懲役（情状により加重・減軽あり） |
| **被害者の保護** | 被害届の提出、被害者支援センターや警察の相談窓口が設置されている |

### 社会的・倫理的側面

- **同意**は明確かつ自由意志に基づくものでなければならない。強圧、脅迫、暴力、欺瞞（嘘）によって得られた同意は無効とされます。  
- **被害者の心理的影響**は深刻で、PTSD（心的外傷後


In [6]:
# ============================================================
# Cell 7: Stage 1 SFT学習
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

# Google Drive上の保存先
STAGE1_OUTPUT_DIR = f"{BASE_DIR}/sft_outputs/stage1"

trainer_stage1 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage1_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=STAGE1_OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=100,
        fp16=False,
        bf16=True,                   # A100対応
        logging_steps=10,
        save_steps=500,
        save_total_limit=3,
        optim="adamw_8bit",          # メモリ効率の良いオプティマイザ
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Stage 1 学習を開始します...")
trainer_stage1.train()
print("Stage 1 学習完了!")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/10000 [00:00<?, ? examples/s]

Stage 1 学習を開始します...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,632,143,872 (0.49% trained)


Step,Training Loss
10,1.715100
20,1.679600
30,1.573900
40,1.509700
50,1.426400
60,1.398200
70,1.356700
80,1.287100
90,1.305500
100,1.271900


Stage 1 学習完了!


In [7]:
# ============================================================
# Cell 8: Stage 1 チェックポイント保存
# ============================================================
STAGE1_ADAPTER_PATH = f"{BASE_DIR}/sft_outputs/stage1_adapter"

model.save_pretrained(STAGE1_ADAPTER_PATH)
tokenizer.save_pretrained(STAGE1_ADAPTER_PATH)

print(f"Stage 1 アダプタを保存しました: {STAGE1_ADAPTER_PATH}")

Stage 1 アダプタを保存しました: /content/drive/MyDrive/NLP_2026/shared/sft_outputs/stage1_adapter


In [8]:
# ============================================================
# Cell 8b: Stage 1 後のキャッシュクリア
# ============================================================
import gc
import torch
import shutil

# Stage 1 のデータセット・Trainer を解放
del trainer_stage1, stage1_dataset, responses_ds
gc.collect()
torch.cuda.empty_cache()

# HuggingFace datasets キャッシュを削除 (Stage 1 データ ~16GB)
hf_cache = "/root/.cache/huggingface/datasets"
if os.path.exists(hf_cache):
    before = shutil.disk_usage("/").used
    shutil.rmtree(hf_cache, ignore_errors=True)
    after = shutil.disk_usage("/").used
    print(f"HF datasets キャッシュ削除: {(before - after) / 1e9:.1f} GB 解放")
else:
    print("HF datasets キャッシュなし")

# GPU メモリ使用量の確認
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU メモリ: {allocated:.1f} GB 使用中 / {reserved:.1f} GB 確保中")

# ディスク空き容量の確認
total, used, free = shutil.disk_usage("/")
print(f"ディスク: {free / 1e9:.1f} GB 空き / {total / 1e9:.1f} GB 合計")

HF datasets キャッシュ削除: 56.0 GB 解放
GPU メモリ: 7.1 GB 使用中 / 7.2 GB 確保中
ディスク: 173.9 GB 空き / 253.1 GB 合計


## Stage 2: 日本語専門タスクSFT

**データセット**: `tokyotech-llm/Swallow-Nemotron-Post-Training-Dataset-v1` (CC BY 4.0)
- Config: `GPT-OSS-Nemotron-Post-Training-Dataset-v1-Ja-202601`
- Splits: `code`, `math`, `stem`

日本語のコード・数学・STEM分野のデータを使用して、Stage 1で獲得した一般的な会話能力の上に専門的なタスク処理能力を追加します。学習率はStage 1よりも低く設定し、獲得済みの能力を保持しつつ新たな知識を学習します。

In [9]:
# ============================================================
# Cell 10: Stage 2 データセット読み込み & 前処理
# ============================================================
from datasets import load_dataset, Dataset
from itertools import islice

# === サンプル数の制限 ===
STAGE2_MAX_SAMPLES = 10000  # 各splitから均等に取得
# ========================

DATASET_NAME = "tokyotech-llm/Swallow-Nemotron-Post-Training-Dataset-v1"
CONFIG_NAME = "GPT-OSS-Nemotron-Post-Training-Dataset-v1-Ja-202601"

splits = ["code", "math", "stem"]
per_split = STAGE2_MAX_SAMPLES // len(splits)  # 各splitから均等取得

print(f"Stage 2: 各splitから {per_split} 件ずつ streaming で取得")

keep_cols = [
    "uuid", "license", "generator", "version", "category",
    "reasoning", "metadata", "output", "conversation",
]

all_rows = []
for split_name in splits:
    print(f"  {split_name}: streaming中...", end=" ")
    try:
        stream = load_dataset(
            DATASET_NAME, name=CONFIG_NAME,
            split=split_name, streaming=True,
        )
        rows = []
        for row in islice(stream, per_split):
            rows.append({k: row[k] for k in keep_cols if k in row})
        all_rows.extend(rows)
        print(f"{len(rows)} 件取得")
    except Exception as e:
        print(f"失敗 - {e}")

stage2_dataset = Dataset.from_list(all_rows)
print(f"\n合計: {len(stage2_dataset)} 件")

# データの確認
print(f"カラム: {stage2_dataset.column_names}")
print(f"\n--- サンプルデータ (1件目) ---")
sample = stage2_dataset[0]
for key in sample:
    val = str(sample[key])
    print(f"  {key}: {val[:200]}{'...' if len(val) > 200 else ''}")

Stage 2: 各splitから 3333 件ずつ streaming で取得
  code: streaming中... 

README.md: 0.00B [00:00, ?B/s]

3333 件取得
  math: streaming中... 3333 件取得
  stem: streaming中... 3333 件取得

合計: 9999 件
カラム: ['uuid', 'license', 'generator', 'version', 'category', 'reasoning', 'metadata', 'output', 'conversation']

--- サンプルデータ (1件目) ---
  uuid: baaca74e-8491-4ae3-bf26-80176647fde0
  license: CC BY 4.0
  generator: gpt-oss-120b
  version: v1
  category: code
  reasoning: on
  metadata: {"source": "codeforces", "dataset": "taco", "index": 985, "split": "train"}
  output: <|channel|>analysis<|message|>We need to solve known Codeforces problem "Display The Number" (CF 1700?). The answer: maximize integer (as decimal string) with given segment budget n, each digit costs:...
  conversation: [{'content': 'You have a large electronic screen which can display up to $998244353$ decimal digits. The digits are displayed in the same way as on different electronic alarm clocks: each place for a ...


In [10]:
# ============================================================
# Cell 10b: Stage 2 データフォーマット変換
# ============================================================


def format_stage2_conversation(example):
    """Stage 2データセットの conversation カラムからチャット形式に変換する。"""
    messages = []

    if "conversation" in example and example["conversation"]:
        for msg in example["conversation"]:
            role = msg.get("role", "")
            content = msg.get("content", "")
            if not content:
                continue
            if role == "user":
                messages.append({"role": "user", "content": content})
            elif role == "assistant":
                messages.append({"role": "assistant", "content": content})

    if not messages:
        return {"text": ""}

    # tokenizer の chat_template を使用してフォーマット
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


print("データをチャットテンプレート形式に変換中...")
stage2_dataset = stage2_dataset.map(
    format_stage2_conversation,
    num_proc=4,
    remove_columns=stage2_dataset.column_names,
)

# 空のサンプルを除去
stage2_dataset = stage2_dataset.filter(lambda x: len(x["text"]) > 0)
print(f"変換完了: {len(stage2_dataset)} 件")

# 変換結果の確認
print(f"\n--- 変換後サンプル (1件目、先頭500文字) ---")
print(stage2_dataset[0]["text"][:500])

データをチャットテンプレート形式に変換中...


Map (num_proc=4):   0%|          | 0/9999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9999 [00:00<?, ? examples/s]

変換完了: 9999 件

--- 変換後サンプル (1件目、先頭500文字) ---


### 指示:
You have a large electronic screen which can display up to $998244353$ decimal digits. The digits are displayed in the same way as on different electronic alarm clocks: each place for a digit consists of $7$ segments which can be turned on and off to compose different digits. The following picture describes how you can display all $10$ decimal digits:

[Image]

As you can see, different digits may require different number of segments to be turned on. For example, if you want to display


In [11]:
# ============================================================
# Cell 11: Stage 2 SFT学習
# ============================================================
# Stage 1のアダプタから継続学習 (モデルは既にLoRA適用済み)

STAGE2_OUTPUT_DIR = f"{BASE_DIR}/sft_outputs/stage2"

trainer_stage2 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage2_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=STAGE2_OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=5e-5,              # Stage 1より低い学習率
        warmup_steps=50,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Stage 2 学習を開始します...")
trainer_stage2.train()
print("Stage 2 学習完了!")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/9999 [00:00<?, ? examples/s]

Stage 2 学習を開始します...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,999 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,632,143,872 (0.49% trained)


Step,Training Loss
10,1.025400
20,1.014400
30,0.997900
40,0.979700
50,0.940700
60,0.923800
70,0.911500
80,0.945700
90,0.943800
100,0.929800


Stage 2 学習完了!


In [12]:
# ============================================================
# Cell 12: 最終モデル保存
# ============================================================
FINAL_ADAPTER_PATH = f"{BASE_DIR}/sft_outputs/final_adapter"
FINAL_MERGED_PATH = f"{BASE_DIR}/sft_outputs/final_merged"  # オプション

# --- LoRAアダプタのみ保存 (軽量) ---
model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print(f"最終アダプタを保存しました: {FINAL_ADAPTER_PATH}")

# --- ベースモデルとマージした完全モデルの保存 (オプション) ---
# 注意: マージ後のモデルは約17GBのディスク容量が必要です
SAVE_MERGED = False  # True に変更するとマージ版も保存

if SAVE_MERGED:
    print("ベースモデルとマージ中...")
    model.save_pretrained_merged(
        FINAL_MERGED_PATH,
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"マージ済みモデルを保存しました: {FINAL_MERGED_PATH}")

print("\n保存完了!")

最終アダプタを保存しました: /content/drive/MyDrive/NLP_2026/shared/sft_outputs/final_adapter

保存完了!


In [13]:
# ============================================================
# Cell 13: 推論テスト
# ============================================================
FastLanguageModel.for_inference(model)  # 推論モードに切替

# テスト用プロンプト
test_prompts = [
    "日本の首都はどこですか？その歴史について簡単に教えてください。",
    "Pythonでフィボナッチ数列を計算する関数を書いてください。",
    "量子コンピュータの基本原理を中学生にもわかるように説明してください。",
]

for i, prompt in enumerate(test_prompts):
    print(f"\n{'='*60}")
    print(f"テスト {i+1}: {prompt}")
    print(f"{'='*60}")

    messages = [
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )

    # 入力部分を除いた生成テキストのみデコード
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    print(f"\n応答:\n{response}")


テスト 1: 日本の首都はどこですか？その歴史について簡単に教えてください。

応答:
**日本の首都は東京です。**  

---

## 東京の歴史（概略）

| 時代 | 主な出来事・特徴 |
|------|----------------|
| **奈良時代（710‑794年）** | 平城京（現在の奈良）が都として建設されたが、徐々に政治・経済の中心地が東へ移動し始める。 |
| **平安時代（794‑1185年）** | 京都に平安京が置かれ、約千年にわたり文化・宗教の中心となる。 |
| **鎌倉時代（1185‑1333年）〜室町時代（1336‑1573年）** | 武士政権が台頭し、政治的中心は京都に残るも、実務上の行政は関東へとシフト。 |
| **戦国時代〜安土桃山時代（1467‑1603年）** | 戦乱が続く中、徳川家康が江戸（現在の東京）を重要視。江戸は商業・物流の拠点として発展。 |
| **江戸時代（1603‑1868年）** | 徳川幕府が江戸を「天下普請」で拡張し、人口が急増。18世紀後半には世界有数の大都市へ成長。 |
| **幕末・明治維新（1868‑1877年）** | 明治天皇が京都から東京に遷都（1868年）。新政府は西洋化・近代化を推進し、首都としてのインフラ整備が急ピッチで行われた。 |
| **20世紀以降** | 第二次世界大戦後の復興と高度経済成長期を経て、現在に至ります。現在も日本の政治・経済・文化の中心として機能しています。 |

> **ポイント**：東京は「遷都」によって正式に首都となったわけではなく、実質的な政治的・経済的中心地が自然に東へ移動した結果、明治時代に公式に首都と定められたという経緯があります。

テスト 2: Pythonでフィボナッチ数列を計算する関数を書いてください。

応答:
**解説（日本語）**  
フィボナッチ数列は「0,1」からはじめて、以降は直前の 2 つの数の和を順に並べたものです。  
代表的な実装方法は次の 3 通りです。

| 方法 | 計算量 | 主な特徴 |
|------|--------|----------|
| **再帰 (recursive)** | `O(2^n)` | コードがシンプルで直感的だが、`n` が大きくなるとすぐに遅すぎる。 |
|